In [2]:
import gc, numpy as np, pandas as pd, torch, torch.nn as nn

DATA = "/kaggle/input/competitions/icecube-neutrinos-in-deep-ice"   # your path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

geometry = pd.read_csv(f"{DATA}/sensor_geometry.csv")
meta = pd.read_parquet(f"{DATA}/train_meta.parquet")
geo = geometry[["x", "y", "z"]].values.astype(np.float32) / 500.0

def angles_to_vec(az, zen):
    return np.stack([np.sin(zen)*np.cos(az), np.sin(zen)*np.sin(az), np.cos(zen)], axis=-1)

def build_features(batch_df, meta_df, maxlen=128):
    p_sensor = batch_df["sensor_id"].values
    p_time   = batch_df["time"].values.astype(np.float32)
    p_charge = batch_df["charge"].values.astype(np.float32)
    p_aux    = batch_df["auxiliary"].values.astype(np.float32)
    N = len(meta_df)
    X = np.zeros((N, maxlen, 6), dtype=np.float32)
    M = np.zeros((N, maxlen), dtype=bool)
    firsts, lasts = meta_df.first_pulse_index.values, meta_df.last_pulse_index.values
    for i in range(N):
        s, e = firsts[i], lasts[i] + 1
        order = np.argsort(p_aux[s:e] * 1e9 + p_time[s:e])[:maxlen]
        idx = np.arange(s, e)[order]
        n = len(idx)
        X[i, :n, 0:3] = geo[p_sensor[idx]]
        X[i, :n, 3]   = (p_time[idx] - p_time[idx].min()) / 1e4
        X[i, :n, 4]   = np.log10(p_charge[idx]) / 3.0
        X[i, :n, 5]   = p_aux[idx]
        M[i, :n] = True
    return X, M

def knn_indices(pos, mask, k):
    d = torch.cdist(pos, pos)
    d = d.masked_fill(~mask[:, None, :], float("inf"))
    return d.topk(k, dim=-1, largest=False).indices

def gather_neighbors(h, idx):
    b = torch.arange(h.shape[0], device=h.device)[:, None, None]
    return h[b, idx]

class EdgeConv(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2*c_in, c_out), nn.LayerNorm(c_out), nn.GELU(),
            nn.Linear(c_out, c_out), nn.GELU())
        self.skip = nn.Linear(c_in, c_out)
    def forward(self, h, idx, nbr_mask, mask):
        hj = gather_neighbors(h, idx)
        hi = h.unsqueeze(2).expand_as(hj)
        msg = self.mlp(torch.cat([hi, hj - hi], dim=-1))
        msg = msg.masked_fill(~nbr_mask.unsqueeze(-1), -1e9)
        out = msg.max(dim=2).values + self.skip(h)
        return out * mask.unsqueeze(-1)

class IceCubeGNN(nn.Module):
    def __init__(self, k=8, hidden=64, n_layers=3):
        super().__init__()
        self.k = k
        self.embed = nn.Sequential(nn.Linear(6, hidden), nn.GELU())
        self.layers = nn.ModuleList([EdgeConv(hidden, hidden) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(2*hidden, 128), nn.GELU(), nn.Linear(128, 3))
    def forward(self, x, mask):
        idx = knn_indices(x[..., :4], mask, self.k)
        b = torch.arange(x.shape[0], device=x.device)[:, None, None]
        nbr_mask = mask[b, idx]
        h = self.embed(x) * mask.unsqueeze(-1)
        for layer in self.layers:
            h = layer(h, idx, nbr_mask, mask)
        m = mask.unsqueeze(-1)
        h_mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
        h_max  = h.masked_fill(~m, -1e9).max(1).values
        out = self.head(torch.cat([h_mean, h_max], dim=-1))
        return out / out.norm(dim=-1, keepdim=True).clamp(min=1e-8)

def angular_loss(pred, true):
    return torch.acos((pred * true).sum(-1).clamp(-1+1e-6, 1-1e-6)).mean()

def cosine_loss(pred, true):
    return (1 - (pred * true).sum(-1)).mean()

@torch.no_grad()
def evaluate(model, X, M, Y, bs=512):
    model.eval()
    total = 0.0
    for i in range(0, len(X), bs):
        total += angular_loss(model(X[i:i+bs], M[i:i+bs]), Y[i:i+bs]).item() * len(X[i:i+bs])
    return total / len(X)

# Fixed validation set: first 10k events of batch 1 (training uses batches 2+)
b1 = pd.read_parquet(f"{DATA}/train/batch_1.parquet")
vm = meta[meta.batch_id == 1].reset_index(drop=True).iloc[:10_000]
Xv, Mv = build_features(b1, vm)
Yv = angles_to_vec(vm.azimuth.values, vm.zenith.values).astype(np.float32)
Xva, Mva, Yva = (torch.from_numpy(a).to(device) for a in (Xv, Mv, Yv))
del b1; gc.collect()
print("ready | val:", Xva.shape, "| device:", device)

ready | val: torch.Size([10000, 128, 6]) | device: cuda


In [4]:
BATCHES = [2, 3,4,5,6]         
EPOCHS_PER_BATCH = 2
BS = 256
N_PER_BATCH = 200_000
steps_per_batch = N_PER_BATCH // BS
total_steps = len(BATCHES) * EPOCHS_PER_BATCH * steps_per_batch

model2 = IceCubeGNN(k=8, hidden=128, n_layers=3).to(device)
opt = torch.optim.AdamW(model2.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=2e-3, total_steps=total_steps, pct_start=0.1)  

for b in BATCHES:
    bdf = pd.read_parquet(f"{DATA}/train/batch_{b}.parquet")
    bm  = meta[meta.batch_id == b].reset_index(drop=True).iloc[:N_PER_BATCH]
    Xb, Mb = build_features(bdf, bm)
    Yb = angles_to_vec(bm.azimuth.values, bm.zenith.values).astype(np.float32)
    del bdf; gc.collect()

    Xg, Mg, Yg = (torch.from_numpy(a).to(device) for a in (Xb, Mb, Yb))
    for ep in range(EPOCHS_PER_BATCH):
        model2.train()
        perm = torch.randperm(len(Xg), device=device)
        run = 0.0
        for s in range(steps_per_batch):
            ids = perm[s*BS:(s+1)*BS]
            pred = model2(Xg[ids], Mg[ids])
            loss = cosine_loss(pred, Yg[ids])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
            opt.step(); sched.step()
            run += angular_loss(pred.detach(), Yg[ids]).item()
        print(f"batch {b} ep {ep+1} | train {run/steps_per_batch:.4f} "
              f"| val {evaluate(model2, Xva, Mva, Yva):.4f}")
    torch.save(model2.state_dict(), f"gnn_stream_b{b}.pt")   # checkpoint
    del Xg, Mg, Yg, Xb, Mb; torch.cuda.empty_cache()

batch 2 ep 1 | train 1.5073 | val 1.3846
batch 2 ep 2 | train 1.3325 | val 1.2973
batch 3 ep 1 | train 1.2359 | val 1.2218
batch 3 ep 2 | train 1.2084 | val 1.2015
batch 4 ep 1 | train 1.1957 | val 1.1848
batch 4 ep 2 | train 1.1780 | val 1.1851
batch 5 ep 1 | train 1.1653 | val 1.1685
batch 5 ep 2 | train 1.1501 | val 1.1565
batch 6 ep 1 | train 1.1406 | val 1.1541
batch 6 ep 2 | train 1.1326 | val 1.1500


In [6]:
import os
print(os.listdir("/kaggle/working"))
print(os.listdir(DATA))
print(os.listdir(f"{DATA}/test") if os.path.exists(f"{DATA}/test") else "no test folder")

['gnn_stream_b5.pt', 'gnn_stream_b4.pt', 'gnn_stream_b6.pt', 'gnn_stream_b2.pt', '.virtual_documents', 'gnn_stream_b3.pt']
['train_meta.parquet', 'sample_submission.parquet', 'test_meta.parquet', 'test', 'sensor_geometry.csv', 'train']
['batch_661.parquet']


In [7]:
def vec_to_angles(v):
    zenith  = np.arccos(np.clip(v[:, 2], -1, 1))
    azimuth = np.arctan2(v[:, 1], v[:, 0]) % (2 * np.pi)
    return azimuth, zenith

test_meta = pd.read_parquet(f"{DATA}/test_meta.parquet")
model2.eval()
rows = []
for b in test_meta.batch_id.unique():
    tm = test_meta[test_meta.batch_id == b].reset_index(drop=True)
    tb = pd.read_parquet(f"{DATA}/test/batch_{b}.parquet")
    Xt, Mt = build_features(tb, tm)
    Xt, Mt = torch.from_numpy(Xt).to(device), torch.from_numpy(Mt).to(device)
    preds = []
    with torch.no_grad():
        for i in range(0, len(Xt), 512):
            preds.append(model2(Xt[i:i+512], Mt[i:i+512]).cpu().numpy())
    az, zen = vec_to_angles(np.concatenate(preds))
    rows.append(pd.DataFrame({"event_id": tm.event_id.values, "azimuth": az, "zenith": zen}))
sub = pd.concat(rows)
sub.to_csv("submission.csv", index=False)
print(sub.head(), len(sub))

   event_id   azimuth    zenith
0      2092  6.095327  0.053907
1      7344  1.452852  0.382638
2      9482  4.419690  1.747404 3
